In [21]:
import re
import pandas as pd
from datetime import datetime

def parse_comparison_results(txt_file, excel_file='comparison_results.xlsx'):
    """
    Đọc file comparison_results.txt và chuyển sang Excel
    """
    
    # Đọc toàn bộ nội dung file
    with open(txt_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Tách các lần so sánh (mỗi khối bắt đầu bằng dấu ====)
    blocks = content.split('='*40)
    
    # Danh sách lưu kết quả
    results = []
    
    for block in blocks:
        if not block.strip():
            continue
            
        # Dictionary lưu thông tin 1 lần so sánh
        data = {}
        
        try:
            img_match = re.search(r"=== THÔNG SỐ CƠ BẢN ẢNH SỐ (.+)===", block)
            data['Ảnh số'] = int(img_match.group(1))
            # Lấy kích thước
            shape1_match = re.search(r'Ảnh 1\s*\(Ảnh gốc\)\s*:\s*\((\d+),\s*(\d+),\s*(\d+)\)', block)
            shape2_match = re.search(r'Ảnh 2\s*\(Ảnh qua xử lý\)\s*:\s*\((\d+),\s*(\d+),\s*(\d+)\)', block)
            if shape1_match:
                data['Chiều cao 1'] = int(shape1_match.group(1))
                data['Chiều rộng 1'] = int(shape1_match.group(2))
            if shape2_match:
                data['Chiều cao 2'] = int(shape2_match.group(1))
                data['Chiều rộng 2'] = int(shape2_match.group(2))
            
            # Tỷ lệ kích thước
            ratio_match = re.search(r'Tỷ lệ kích thước: ([\d.]+)x([\d.]+)', block)
            if ratio_match:
                data['Tỷ lệ H'] = float(ratio_match.group(1))
                data['Tỷ lệ W'] = float(ratio_match.group(2))
            
            # Số pixel
            pixels_match = re.search(r'Số pixel - Ảnh 1: ([\d,]+) \| Ảnh 2: ([\d,]+)', block)
            if pixels_match:
                data['Số pixel 1'] = int(pixels_match.group(1).replace(',', ''))
                data['Số pixel 2'] = int(pixels_match.group(2).replace(',', ''))
            
            # Giá trị trung bình BGR - Ảnh 1
            bgr1_match = re.search(r'Giá trị TB \(BGR\) - Ảnh 1:\s*\[\s*([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*\]', block)
            if bgr1_match:
                data['BGR_B1'] = float(bgr1_match.group(1))
                data['BGR_G1'] = float(bgr1_match.group(2))
                data['BGR_R1'] = float(bgr1_match.group(3))
            
            # Giá trị trung bình BGR - Ảnh 2
            bgr2_match = re.search(r'Giá trị TB \(BGR\) - Ảnh 2:\s*\[\s*([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*\]', block)
            if bgr2_match:
                data['BGR_B2'] = float(bgr2_match.group(1))
                data['BGR_G2'] = float(bgr2_match.group(2))
                data['BGR_R2'] = float(bgr2_match.group(3))
            
            # Độ sáng
            brightness_match = re.search(r'Độ sáng TB - Ảnh 1: ([\d.]+) \| Ảnh 2: ([\d.]+)', block)
            if brightness_match:
                data['Độ sáng 1'] = float(brightness_match.group(1))
                data['Độ sáng 2'] = float(brightness_match.group(2))
            
            # Độ tương phản
            contrast_match = re.search(r'Độ tương phản - Ảnh 1: ([\d.]+) \| Ảnh 2: ([\d.]+)', block)
            if contrast_match:
                data['Độ tương phản 1'] = float(contrast_match.group(1))
                data['Độ tương phản 2'] = float(contrast_match.group(2))

            # Tương quan histogram
            hist_match = re.search(r'Tương quan histogram:\s*([\d.]+)', block)
            if not hist_match:
                hist_match = re.search(r'histogram:\s*([\d.]+)', block, re.IGNORECASE)
            if hist_match:
                data['Tương quan Histogram'] = float(hist_match.group(1))

            # SSIM
            ssim_match = re.search(r'SSIM Score: ([\d.]+)', block)
            if ssim_match:
                data['SSIM'] = float(ssim_match.group(1))
            
            # MSE
            mse_match = re.search(r'MSE: ([\d.]+)', block)
            if mse_match:
                data['MSE'] = float(mse_match.group(1))
            
            # PSNR
            psnr_match = re.search(r'PSNR: ([\d.]+)', block)
            if psnr_match:
                data['PSNR'] = float(psnr_match.group(1))
            
            # Thêm vào danh sách nếu có dữ liệu
            if len(data) > 2:  # Ít nhất có thời gian và tên ảnh
                results.append(data)
        
        except Exception as e:
            print(f"Lỗi khi xử lý block: {e}")
            continue
    
    # Tạo DataFrame
    df = pd.DataFrame(results)
    
    # Sắp xếp cột theo thứ tự logic
    column_order = [
        'Ảnh số', 'Ảnh 1 (Ảnh gốc)', 'Ảnh 2 (Ảnh qua xử lý)',
        'Chiều cao 1', 'Chiều rộng 1', 'Số pixel 1',
        'Chiều cao 2', 'Chiều rộng 2', 'Số pixel 2',
        'Tỷ lệ H', 'Tỷ lệ W',
        'BGR_B1', 'BGR_G1', 'BGR_R1',
        'BGR_B2', 'BGR_G2', 'BGR_R2',
        'Độ sáng 1', 'Độ sáng 2',
        'Độ tương phản 1', 'Độ tương phản 2',
        'Tương quan Histogram',
        'SSIM', 'MSE', 'PSNR'
    ]
    
    # Chỉ giữ các cột có trong dữ liệu
    existing_columns = [col for col in column_order if col in df.columns]
    df = df[existing_columns]
    
    # Xuất ra Excel với format đẹp
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='Comparison Results', index=False)
        
        # Lấy worksheet để format
        worksheet = writer.sheets['Comparison Results']
        
        # Tự động điều chỉnh độ rộng cột
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            worksheet.column_dimensions[column_letter].width = adjusted_width
    
    print(f"✓ Đã chuyển đổi thành công!")
    print(f"  - Số lần so sánh: {len(results)}")
    print(f"  - File Excel: {excel_file}")
    print(f"  - Số cột: {len(df.columns)}")
    
    return df


In [22]:
parse_comparison_results('Thong_so_co_ban.txt')

✓ Đã chuyển đổi thành công!
  - Số lần so sánh: 1254
  - File Excel: comparison_results.xlsx
  - Số cột: 23


,Ảnh số,Chiều cao 1,Chiều rộng 1,Số pixel 1,Chiều cao 2,Chiều rộng 2,Số pixel 2,Tỷ lệ H,Tỷ lệ W,BGR_B1,...,BGR_G2,BGR_R2,Độ sáng 1,Độ sáng 2,Độ tương phản 1,Độ tương phản 2,Tương quan Histogram,SSIM,MSE,PSNR
0,1,800,1200,960000,1600,2400,3840000,0.5,0.5,28.540604,...,59.915052,47.123485,52.70,52.50,35.69,35.45,1.0,0.9951,5.19,40.98
1,2,800,1200,960000,1600,2400,3840000,0.5,0.5,182.015230,...,186.103114,190.812951,187.16,186.97,49.41,49.40,1.0,0.9970,5.29,40.90
2,3,800,1200,960000,1600,2400,3840000,0.5,0.5,23.386837,...,25.577101,40.988696,30.10,30.01,29.36,29.02,1.0,0.9919,7.81,39.20
3,4,800,1200,960000,1600,2400,3840000,0.5,0.5,79.954250,...,78.884496,88.365147,81.87,81.83,60.01,59.99,1.0,0.9991,0.40,52.11
4,5,800,1200,960000,1600,2400,3840000,0.5,0.5,95.558984,...,133.351376,173.346992,141.05,141.01,68.58,68.60,1.0,0.9997,0.21,54.86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1249,1250,800,1200,960000,1600,2400,3840000,0.5,0.5,98.299501,...,101.649043,112.141102,104.51,104.40,94.16,94.19,1.0,0.9975,0.60,50.35
1250,1251,799,1200,958800,1598,2400,3835200,0.5,0.5,102.633414,...,110.678121,118.172654,112.15,111.98,88.86,88.84,1.0,0.9927,4.84,41.28
1251,1252,800,1200,960000,1600,2400,3840000,0.5,0.5,164.787986,...,167.306539,170.607682,168.07,168.00,98.28,98.37,1.0,0.9986,1.02,48.03
1252,1253,800,1200,960000,1600,2400,3840000,0.5,0.5,88.331803,...,99.259080,105.832121,100.11,99.98,76.99,76.98,1.0,0.9976,0.66,49.95
